In [1]:
import pandas as pd
df = pd.read_csv("dataset_with_class.csv")
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

# 1️⃣ قراءة الداتا (افترض ان df موجودة)

# 2️⃣ اختيار Features و Label
features = ["NSS", "heat right", "heat left", "cold right", "cold left",
            "BMI baseline", "Age baseline", "HbA1c baseline"]
X = df[features].apply(pd.to_numeric, errors='coerce')
y = df['Neuropathy']

# 3️⃣ تنظيف Missing Values عن طريق تعويضهم بالمتوسط
X = X.fillna(X.mean())
y = y.fillna(y.mode()[0])  # في حالة Missing labels

# 4️⃣ Feature Engineering (ممكن تزودي الأعمدة الجديدة)
X['heat_avg'] = (X['heat right'] + X['heat left']) / 2
X['cold_avg'] = (X['cold right'] + X['cold left']) / 2
X['BMI_Age'] = X['BMI baseline'] / X['Age baseline']

# 5️⃣ Standard Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6️⃣ تطبيق SMOTE لتوليد ~100 مثال لكل Class
class_counts = y.value_counts()
sampling_strategy = {cls: max(100, class_counts[cls]) for cls in class_counts.index}
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)
X_res, y_res = smote.fit_resample(X_scaled, y)

# 7️⃣ تقسيم البيانات للتدريب والاختبار
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res
)

# 8️⃣ Random Forest مع Hyperparameter Tuning باستخدام GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

# 9️⃣ توقع على بيانات الاختبار
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:,1]

#  🔹 تقييم الموديل
accuracy = accuracy_score(y_test, y_pred)
print("Best Random Forest Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 10️⃣ Feature Importance
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nFeature Importance:")
print(importance)

# 11️⃣ حفظ النتائج لكل مريض في CSV
results_df = pd.DataFrame(X_test, columns=X.columns)
results_df['Actual Class'] = y_test
results_df['Predicted Class'] = y_pred
results_df['Predicted Probability Neuropathy'] = y_prob
results_df.to_csv("random_forest_results.csv", index=False)
print("\nتم حفظ النتائج في ملف: random_forest_results.csv")


Best Random Forest Accuracy: 0.9

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.90      0.90        20
           1       0.90      0.90      0.90        20

    accuracy                           0.90        40
   macro avg       0.90      0.90      0.90        40
weighted avg       0.90      0.90      0.90        40


Confusion Matrix:
 [[18  2]
 [ 2 18]]

Feature Importance:
           Feature  Importance
0              NSS    0.154208
10         BMI_Age    0.130112
5     BMI baseline    0.109807
1       heat right    0.108037
7   HbA1c baseline    0.093481
4        cold left    0.080370
3       cold right    0.078825
8         heat_avg    0.076723
9         cold_avg    0.066562
2        heat left    0.059091
6     Age baseline    0.042786

تم حفظ النتائج في ملف: random_forest_results.csv
